In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(len(filenames))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

0
0
0
0
2
2
4
1
0
1
1655
1655
0
4952
4952
0
1
14896
14896


In [2]:
import sys
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [3]:
!pip install -q --no-deps ultralytics==8.3.0

import ultralytics
print('Ultralytics:', ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.3/881.3 kB 14.5 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.3.0


In [4]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/voc_project')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

files = [path for path in INPUT_ROOT.rglob('*') if path.is_file()]
print(f'Files found: {len(files)}')
for path in files[:100]:
    print(path)

archives = [path for path in files if path.suffix.lower() == '.zip']
train_dirs = list(INPUT_ROOT.rglob('train/images'))
val_dirs = list(INPUT_ROOT.rglob('val/images'))
test_dirs = list(INPUT_ROOT.rglob('test/images'))

print('ZIP archives:', archives)
print('Existing train directories:', train_dirs)
print('Existing validation directories:', val_dirs)
print('Existing test directories:', test_dirs)

if not archives and not (train_dirs and val_dirs and test_dirs):
    raise FileNotFoundError(
        'Attached input was found, but it contains neither ZIP archives nor '
        'train/images, val/images, and test/images directories.'
    )

Files found: 43017
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_checkpoints/last.pt
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_checkpoints/best.pt
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/README.md
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/requirements.txt
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/scripts/train.py
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/scripts/evaluate.py
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/scripts/prepare_dataset.py
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/scripts/predict_baseline.py
/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_code/conf

In [5]:
for archive in archives:
    print('Extracting:', archive.name)
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(WORK_ROOT)
print('Extraction complete.')

Extraction complete.


In [6]:
from pathlib import Path
import zipfile

# Extract attached archives if the working directory does not contain the splits yet.
WORK_ROOT.mkdir(parents=True, exist_ok=True)
existing_train = list(WORK_ROOT.rglob('train/images'))
if not existing_train:
    archives = list(INPUT_ROOT.rglob('*.zip'))
    print('Archives to extract:', archives)
    for archive in archives:
        with zipfile.ZipFile(archive) as zip_file:
            zip_file.extractall(WORK_ROOT)

# Search both the extracted working directory and Kaggle input.
search_roots = [WORK_ROOT, INPUT_ROOT]
train_dirs = []
val_dirs = []
test_dirs = []
for root in search_roots:
    train_dirs.extend(root.rglob('train/images'))
    val_dirs.extend(root.rglob('val/images'))
    test_dirs.extend(root.rglob('test/images'))

print('Train:', train_dirs)
print('Validation:', val_dirs)
print('Test:', test_dirs)

if not train_dirs or not val_dirs or not test_dirs:
    raise FileNotFoundError(
        'Train, validation, or test images were not found. '
        'Run the input diagnostic cell and confirm the Kaggle dataset is attached.'
    )

train_dir = train_dirs[0]
val_dir = val_dirs[0]
test_dir = test_dirs[0]
DATA_ROOT = train_dir.parent.parent

print('Dataset root:', DATA_ROOT)
print('Train images:', len(list(train_dir.glob('*.jpg'))))
print('Validation images:', len(list(val_dir.glob('*.jpg'))))
print('Test images:', len(list(test_dir.glob('*.jpg'))))

Archives to extract: []
Train: [PosixPath('/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_train/train/images')]
Validation: [PosixPath('/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/val/images')]
Test: [PosixPath('/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images')]
Dataset root: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_train
Train images: 14896
Validation images: 1655
Test images: 4952


In [7]:
checkpoint_roots = [WORK_ROOT, INPUT_ROOT]
checkpoint_files = []
for root in checkpoint_roots:
    checkpoint_files.extend(root.rglob('*.pt'))

print('Checkpoints:', checkpoint_files)
last_files = [path for path in checkpoint_files if path.name == 'last.pt']
best_files = [path for path in checkpoint_files if path.name == 'best.pt']
CHECKPOINT = last_files[0] if last_files else best_files[0] if best_files else None

if CHECKPOINT is None:
    raise FileNotFoundError(
        'No best.pt or last.pt checkpoint was found in Kaggle input or working storage.'
    )

print('Using checkpoint:', CHECKPOINT)

Checkpoints: [PosixPath('/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_checkpoints/last.pt'), PosixPath('/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_checkpoints/best.pt')]
Using checkpoint: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_checkpoints/last.pt


In [8]:
import yaml

DATA_YAML = Path('/kaggle/working/voc_kaggle.yaml')

# The train, validation, and test archives are separate Kaggle inputs.
# Use absolute paths so Ultralytics does not assume one shared dataset root.
config = {
    'path': '/',
    'train': str(train_dir),
    'val': str(val_dir),
    'test': str(test_dir),
    'names': [
        'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
        'bus', 'car', 'cat', 'chair', 'cow',
        'diningtable', 'dog', 'horse', 'motorbike', 'person',
        'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
    ]
}

DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(DATA_YAML.read_text())

path: /
train: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_train/train/images
val: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/val/images
test: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images
names:
- aeroplane
- bicycle
- bird
- boat
- bottle
- bus
- car
- cat
- chair
- cow
- diningtable
- dog
- horse
- motorbike
- person
- pottedplant
- sheep
- sofa
- train
- tvmonitor



In [9]:
from ultralytics.data.utils import check_det_dataset
dataset_info = check_det_dataset(str(DATA_YAML))
print('Train:', dataset_info['train'])
print('Validation:', dataset_info['val'])
print('Test:', dataset_info['test'])
print('Dataset validation passed.')

100%|██████████| 755k/755k [00:00<00:00, 18.2MB/s]

Train: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_train/train/images
Validation: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/val/images
Test: /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images
Dataset validation passed.


In [11]:
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

import wandb
wandb.init(mode='disabled')

# Kaggle's preinstalled ray version is missing an attribute the Ultralytics
# raytune callback expects; patch it since Ray Tune is not being used here.
import ray.train._internal.session as ray_session
if not hasattr(ray_session, '_get_session'):
    ray_session._get_session = lambda: None

from ultralytics import YOLO

model = YOLO(str(CHECKPOINT))
model.train(
    data=str(DATA_YAML),
    epochs=30,
    resume=False,
    imgsz=640,
    batch=8,
    workers=2,
    device=0,
    patience=8,
    amp=True,
    save_period=5,
    project='/kaggle/working/runs/detect',
    name='voc_yolov8n_t4',
    seed=42,
    deterministic=True,
    cache=False
)

New https://pypi.org/project/ultralytics/8.4.154 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_checkpoints/last.pt, data=/kaggle/working/voc_kaggle.yaml, epochs=30, time=None, patience=8, batch=8, imgsz=640, save=True, save_period=5, cache=False, device=0, workers=2, project=/kaggle/working/runs/detect, name=voc_yolov8n_t45, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=

train: Scanning /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_train/train/labels... 14896 images, 0 backgrounds, 0 corrupt: 100%|██████████| 14896/14896 [00:12<00:00, 1201.32it/s]


train: WARNING ⚠️ Cache directory /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_train/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/val/labels... 1655 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1655/1655 [00:01<00:00, 1207.66it/s]

val: WARNING ⚠️ Cache directory /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/val is not writeable, cache not saved.


...

                   all       1655       3978      0.819      0.667       0.77      0.575



30 epochs completed in 1.667 hours.
Optimizer stripped from /kaggle/working/runs/detect/voc_yolov8n_t45/weights/last.pt, 5.6MB
Optimizer stripped from /kaggle/working/runs/detect/voc_yolov8n_t45/weights/best.pt, 5.6MB

Validating /kaggle/working/runs/detect/voc_yolov8n_t45/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 186 layers, 2,688,268 parameters, 0 gradients


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 104/104 [00:11<00:00,  9.18it/s]


                   all       1655       3978      0.818      0.668       0.77      0.576
             aeroplane        102        130      0.843      0.792      0.872      0.677
               bicycle         73        104      0.893      0.807      0.866      0.645
                  bird        107        172      0.818      0.564      0.695      0.476
                  boat         64        107      0.814      0.523      0.619      0.433
                bottle         96        186      0.766      0.559      0.631      0.412
                   bus         66         90      0.876      0.707      0.834      0.702
                   car        197        346      0.869      0.746      0.848      0.628
                   cat        140        156      0.852      0.808      0.894      0.734
                 chair        132        276      0.752      0.462       0.61      0.428
                   cow         43         84      0.741      0.571       0.74      0.551
           diningtabl

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a1b8f6e9bb0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043

In [12]:
best_model = YOLO('/kaggle/working/runs/detect/voc_yolov8n_t45/weights/best.pt')

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=640,
    batch=8,
    device=0,
    conf=0.001,
    iou=0.7,
    project='/kaggle/working/runs/eval',
    name='voc_test',
    plots=True
)

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 186 layers, 2,688,268 parameters, 0 gradients


val: Scanning /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/labels... 4952 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4952/4952 [00:18<00:00, 262.87it/s]


val: WARNING ⚠️ Cache directory /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test is not writeable, cache not saved.


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 619/619 [00:36<00:00, 16.94it/s]


                   all       4952      12032      0.802      0.698      0.791      0.578
             aeroplane        204        285      0.923      0.715      0.842       0.61
               bicycle        239        337      0.902      0.774      0.882      0.651
                  bird        282        459       0.81      0.654      0.764      0.515
                  boat        172        263      0.738      0.589      0.684      0.436
                bottle        212        469      0.828      0.596      0.681      0.444
                   bus        174        213      0.835      0.789      0.876      0.737
                   car        721       1201      0.884      0.817      0.911      0.715
                   cat        322        358      0.841      0.779      0.855      0.669
                 chair        417        756      0.672      0.532      0.609      0.413
                   cow        127        244       0.73      0.635      0.776      0.575
           diningtabl

In [13]:
import shutil
from pathlib import Path

EVAL_DIR = Path('/kaggle/working/runs/eval')
ZIP_PATH = Path('/kaggle/working/voc_eval_results')

if not EVAL_DIR.exists():
    raise FileNotFoundError(f'{EVAL_DIR} does not exist. Run test evaluation first.')

archive_path = shutil.make_archive(str(ZIP_PATH), 'zip', root_dir=str(EVAL_DIR))
print('Created:', archive_path)
print('Size (MB):', round(Path(archive_path).stat().st_size / (1024 * 1024), 2))

Created: /kaggle/working/voc_eval_results.zip
Size (MB): 3.84


In [14]:
best_model.predict(
    source=str(test_dir),
    imgsz=640,
    batch=8,
    device=0,
    conf=0.25,
    iou=0.7,
    save=True,
    project='/kaggle/working/runs/predict',
    name='voc_test_predictions'
)


image 1/4952 /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images/VOC2007_000001.jpg: 640x640 1 dog, 1 person, 5.2ms
image 2/4952 /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images/VOC2007_000002.jpg: 640x640 1 train, 5.2ms
image 3/4952 /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images/VOC2007_000003.jpg: 640x640 (no detections), 5.2ms
image 4/4952 /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images/VOC2007_000004.jpg: 640x640 8 cars, 5.2ms
image 5/4952 /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images/VOC2007_000006.jpg: 640x640 2 chairs, 1 diningtable, 5.2ms
image 6/4952 /kaggle/input/datasets/rituparnochatterjee/pascal-voc-yolov8n-transfer-learning/kaggle_val_test/test/images/VOC2007_000008.jpg: 640x640

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}
 obb: None
 orig_img: array([[[  0,   1,   1],
         [  0,   1,   1],
         [  0,   1,   1],
         ...,
         [  3,   4,   2],
         [  3,   4,   2],
         [  2,   3,   1]],
 
        [[  0,   1,   1],
         [  0,   1,   1],
         [  0,   1,   1],
         ...,
         [  4,   5,   3],
         [  3,   4,   2],
         [  2,   3,   1]],
 
        [[  0,   1,   1],
         [  0,   1,   1],
         [  0,   1,   1],
         ...,
         [  5,   6,   4],
         [  3,   4,   2],
         [  2,   2,   2]],
 
        ...,
 
        [[ 95,  54,  45],
     

In [15]:
import shutil
from pathlib import Path

PREDICT_DIR = Path('/kaggle/working/runs/predict')
ZIP_PATH = Path('/kaggle/working/voc_predictions')

if not PREDICT_DIR.exists():
    raise FileNotFoundError(f'{PREDICT_DIR} does not exist. Run prediction generation first.')

archive_path = shutil.make_archive(str(ZIP_PATH), 'zip', root_dir=str(PREDICT_DIR))
print('Created:', archive_path)
print('Size (MB):', round(Path(archive_path).stat().st_size / (1024 * 1024), 2))

Created: /kaggle/working/voc_predictions.zip
Size (MB): 434.47
